# DATATHON 2026 — Sales Forecasting Pipeline v2
**Team:** Async Sleepers / VinIntelligent  
**Horizon:** 548 days (2023-01-01 → 2024-07-01)  
**Strategy:** LightGBM ×3 + Prophet + N-BEATS → SLSQP ensemble  
**Runtime:** T4 GPU từ đầu (chọn T4 trước khi run, không cần đổi runtime giữa chừng)

### Cách chạy
1. Runtime → Change runtime type → **T4 GPU** → Save  
2. Run Cell 1 (install)  
3. Run Cell 2 (upload `sales.csv` và `sample_submission.csv`)  
4. **Ctrl+F9** — run all cells  
5. Kết quả lưu tự động vào `MyDrive/datathon/`


In [ ]:
# Cell 1 — Install (run once)
!pip install lightgbm prophet neuralforecast torch shap --quiet
print('✅ Install complete.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
✅ Install

In [ ]:
# Cell 2 — Upload data files vào Colab session
from google.colab import files
print('Upload sales.csv ...')
files.upload()
print('Upload sample_submission.csv ...')
files.upload()
import os
print('Files in /content:', [f for f in os.listdir('/content') if f.endswith('.csv')])


Upload sales.csv ...


Saving sales.csv to sales.csv
Upload sample_submission.csv ...


Saving sample_submission.csv to sample_submission.csv
Files in /content: ['sales.csv', 'sample_submission.csv']


In [ ]:
# Cell 3 — Imports + Constants  [giữ nguyên cấu trúc Cell 2 gốc, đổi OUTPUT_DIR]
from __future__ import annotations
import os, sys, pickle, warnings, calendar, math
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, roc_auc_score
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Paths ──────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/')
DATA_DIR          = Path('/content')
OUTPUT_DIR        = DRIVE_PROJECT_DIR / 'datathon'   # output → MyDrive/datathon
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_required = ['sales.csv', 'sample_submission.csv']
_missing  = [f for f in _required if not (DATA_DIR / f).exists()]
if _missing:
    raise FileNotFoundError(f'Thiếu files: {_missing} — chạy lại Cell 2 để upload')
print(f'✅ Files OK: {_required}')
print(f'✅ Output dir: {OUTPUT_DIR}')

# ── Constants ──────────────────────────────────────────────────────────────
HORIZON = 548

FEATURE_COLS = [
    'day_of_week', 'month', 'day_of_month', 'is_payday_window',
    'dist_to_payday', 'is_weekend', 'days_to_tet', 'is_tet_buildup',
    'is_tet_peak', 'is_tet_holiday', 'is_tet_reopening', 'is_gift_peak',
    'is_travel_peak', 'is_year_end_festive', 'is_ghost_month',
    'dist_to_nearest_holiday', 'shopping_festival_score',
    'days_to_next_big_sale', 'is_sale_leadup', 'is_black_friday',
    'is_cyber_monday', 'rev_lag_364', 'cogs_lag_364', 'rev_lag_728',
    'cogs_lag_728', 'rev_roll_mean_28_lag_728', 'cogs_roll_mean_28_lag_728',
    'stat_rev_mean_dow_month', 'stat_cogs_mean_dow_month',
    'stat_rev_std_month', 'stat_cogs_std_month',
    'stat_rev_median_day', 'stat_cogs_median_day',
    'stat_rev_yoy_growth_month', 'stat_cogs_yoy_growth_month',
    'sin_annual_1', 'cos_annual_1',
    'is_pre_lockdown', 'is_lockdown', 'is_post_lockdown',
    'is_tiktok_era', 'year',
]
assert len(FEATURE_COLS) == 42, f'Expected 42 features, got {len(FEATURE_COLS)}'
print(f'✅ Feature count: {len(FEATURE_COLS)}')


Mounted at /content/drive
✅ Files OK: ['sales.csv', 'sample_submission.csv']
✅ Output dir: /content/drive/MyDrive/datathon
✅ Feature count: 42


In [ ]:
# Cell 4 — Feature engineering helpers
TET_DATES = {
    2013: pd.Timestamp('2013-02-10'), 2014: pd.Timestamp('2014-01-31'),
    2015: pd.Timestamp('2015-02-19'), 2016: pd.Timestamp('2016-02-08'),
    2017: pd.Timestamp('2017-01-28'), 2018: pd.Timestamp('2018-02-16'),
    2019: pd.Timestamp('2019-02-05'), 2020: pd.Timestamp('2020-01-25'),
    2021: pd.Timestamp('2021-02-12'), 2022: pd.Timestamp('2022-02-01'),
    2023: pd.Timestamp('2023-01-22'), 2024: pd.Timestamp('2024-02-10'),
}
HOLIDAY_YEARS = range(2012, 2025)
COVID_WINDOWS = {
    'is_pre_lockdown':  (pd.Timestamp('2021-05-09'), pd.Timestamp('2021-05-22')),
    'is_lockdown':      (pd.Timestamp('2021-05-23'), pd.Timestamp('2021-10-01')),
    'is_post_lockdown': (pd.Timestamp('2021-10-02'), pd.Timestamp('2021-11-01')),
}

def _jd_from_date(day, month, year):
    a = (14 - month) // 12; y = year + 4800 - a; m = month + 12 * a - 3
    jd = day + ((153*m+2)//5) + 365*y + (y//4) - (y//100) + (y//400) - 32045
    if jd < 2299161:
        jd = day + ((153*m+2)//5) + 365*y + (y//4) - 32083
    return jd

def _new_moon(k):
    t=k/1236.85; t2=t*t; t3=t2*t; dr=math.pi/180
    jd1=2415020.75933+29.53058868*k+0.0001178*t2-0.000000155*t3
    jd1+=0.00033*math.sin((166.56+132.87*t-0.009173*t2)*dr)
    m=359.2242+29.10535608*k-0.0000333*t2-0.00000347*t3
    mp=306.0253+385.81691806*k+0.0107306*t2+0.00001236*t3
    f=21.2964+390.67050646*k-0.0016528*t2-0.00000239*t3
    c1=(0.1734-0.000393*t)*math.sin(m*dr)+0.0021*math.sin(2*dr*m)
    c1-=0.4068*math.sin(mp*dr)+0.0161*math.sin(dr*2*mp)-0.0004*math.sin(dr*3*mp)
    c1+=0.0104*math.sin(dr*2*f)-0.0051*math.sin(dr*(m+mp))
    c1-=0.0074*math.sin(dr*(m-mp))+0.0004*math.sin(dr*(2*f+m))
    c1-=0.0004*math.sin(dr*(2*f-m))-0.0006*math.sin(dr*(2*f+mp))
    c1+=0.0010*math.sin(dr*(2*f-mp))+0.0005*math.sin(dr*(2*mp+m))
    dt = (-0.000278+0.000265*t+0.000262*t2) if t >= -11 else (0.001+0.000839*t+0.0002261*t2-0.00000845*t3-0.000000081*t*t3)
    return jd1+c1-dt

def _get_new_moon_day(k, tz): return int(_new_moon(k)+0.5+tz/24)

def _get_sun_longitude(dn, tz):
    t=(dn-2451545.5-tz/24)/36525; t2=t*t; dr=math.pi/180
    m=357.52910+35999.05030*t-0.0001559*t2; l0=280.46645+36000.76983*t+0.0003032*t2
    dl=(1.914600-0.004817*t-0.000014*t2)*math.sin(dr*m)+(0.019993-0.000101*t)*math.sin(dr*2*m)+0.000290*math.sin(dr*3*m)
    l=(l0+dl)*dr; l=l-math.pi*2*math.floor(l/(math.pi*2))
    return int(l/math.pi*6)

def _get_lunar_month_11(year, tz):
    off=_jd_from_date(31,12,year)-2415021; k=int(off/29.530588853)
    nm=_get_new_moon_day(k,tz)
    if _get_sun_longitude(nm,tz)>=9: nm=_get_new_moon_day(k-1,tz)
    return nm

def _get_leap_month_offset(a11, tz):
    k=int(0.5+(a11-2415021.076998695)/29.530588853); last=0; i=1
    arc=_get_sun_longitude(_get_new_moon_day(k+i,tz),tz)
    while arc!=last and i<14:
        last=arc; i+=1; arc=_get_sun_longitude(_get_new_moon_day(k+i,tz),tz)
    return i-1

def _solar_to_lunar(day, month, year, tz=7.0):
    dn=_jd_from_date(day,month,year); k=int((dn-2415021.076998695)/29.530588853)
    ms=_get_new_moon_day(k+1,tz)
    if ms>dn: ms=_get_new_moon_day(k,tz)
    a11=_get_lunar_month_11(year,tz); b11=a11
    if a11>=ms: ly=year; a11=_get_lunar_month_11(year-1,tz)
    else: ly=year+1; b11=_get_lunar_month_11(year+1,tz)
    ld=dn-ms+1; diff=int((ms-a11)/29); lm=diff+11; ll=0
    if b11-a11>365:
        lmoff=_get_leap_month_offset(a11,tz)
        if diff>=lmoff: lm=diff+10; ll=(1 if diff==lmoff else 0)
    if lm>12: lm-=12
    if lm>=11 and diff<4: ly-=1
    return ld, lm, ly, ll

def _get_black_friday(year):
    fd=pd.Timestamp(year=year,month=11,day=1)
    return fd+pd.Timedelta(days=(3-fd.dayofweek)%7+21+1)

def _get_cyber_monday(year): return _get_black_friday(year)+pd.Timedelta(days=3)

def _calc_dist_to_payday(date, payday=28):
    if date.day<=payday: return payday-date.day
    return (calendar.monthrange(date.year,date.month)[1]-date.day)+payday

def _calc_days_to_tet(date):
    cands=[t for t in TET_DATES.values() if abs((date-t).days)<=366] or list(TET_DATES.values())
    return (date-min(cands,key=lambda t: abs((date-t).days))).days

def _is_gift_peak(date):
    for m,d in ((2,14),(3,8),(10,20)):
        if 2<=(pd.Timestamp(year=date.year,month=m,day=d)-date).days<=7: return True
    return False

def _is_travel_peak(date):
    for m,d in ((4,30),(9,2)):
        if 3<=(pd.Timestamp(year=date.year,month=m,day=d)-date).days<=10: return True
    return False

def _is_ghost_month(date):
    _,lm,_,_=_solar_to_lunar(date.day,date.month,date.year)
    return lm==7

def _calc_dist_to_nearest_holiday(date):
    evs=[(1,1),(2,14),(3,8),(4,30),(9,2),(10,20),(12,24)]
    dists=[]
    for m,d in evs:
        c=pd.Timestamp(year=date.year,month=m,day=d)
        if c<date: c=pd.Timestamp(year=date.year+1,month=m,day=d)
        dists.append((c-date).days)
    return min(dists)

def _calc_shopping_festival_score(date):
    md=(date.month,date.day); y=date.year
    if md==(11,11) and y>=2017: return 3
    if md==(12,12) and y>=2014: return 3
    if md==(9,9)  and y>=2016: return 2
    if md==(10,10) and y>=2016: return 2
    if date.month==date.day and 1<=date.day<=8: return 1
    return 0

def _calc_days_to_next_big_sale(date):
    sales_events=[pd.Timestamp(year=date.year,month=m,day=d)
                  for m,d in [(9,9),(10,10),(11,11),(12,12)]]
    sales_events+=[_get_black_friday(date.year),_get_cyber_monday(date.year)]
    future=[s for s in sales_events if s>=date]
    if future: return min((s-date).days for s in future)
    ny=[pd.Timestamp(year=date.year+1,month=m,day=d) for m,d in [(9,9),(10,10),(11,11),(12,12)]]
    ny+=[_get_black_friday(date.year+1),_get_cyber_monday(date.year+1)]
    return min((s-date).days for s in ny)

def _create_holidays_df():
    rows=[]
    for t in TET_DATES.values():
        rows.append({'ds':t,'holiday':'tet','lower_window':-21,'upper_window':10})
    for yr in HOLIDAY_YEARS:
        rows+=[
            {'ds':pd.Timestamp(yr,11,11),'holiday':'11_11','lower_window':-3,'upper_window':1},
            {'ds':pd.Timestamp(yr,12,12),'holiday':'12_12','lower_window':-3,'upper_window':1},
            {'ds':pd.Timestamp(yr,9,9), 'holiday':'9_9','lower_window':-2,'upper_window':1},
            {'ds':_get_black_friday(yr),'holiday':'black_friday','lower_window':-2,'upper_window':2},
            {'ds':_get_cyber_monday(yr),'holiday':'cyber_monday','lower_window':-1,'upper_window':1},
            {'ds':pd.Timestamp(yr,2,14),'holiday':'valentine','lower_window':-7,'upper_window':0},
            {'ds':pd.Timestamp(yr,3,8), 'holiday':'womens_day','lower_window':-7,'upper_window':0},
            {'ds':pd.Timestamp(yr,10,20),'holiday':'vn_womens_day','lower_window':-7,'upper_window':0},
            {'ds':pd.Timestamp(yr,4,30),'holiday':'reunification','lower_window':-10,'upper_window':0},
            {'ds':pd.Timestamp(yr,9,2), 'holiday':'national_day','lower_window':-10,'upper_window':0},
        ]
    return pd.DataFrame(rows).sort_values(['ds','holiday']).reset_index(drop=True)

def _create_covid_regressors(dates):
    s=pd.Series(pd.to_datetime(dates))
    return pd.DataFrame({col: s.between(start,end).astype(int).values
                         for col,(start,end) in COVID_WINDOWS.items()})

def _compute_statistics(df):
    b=df.copy()
    b['dow']=b['Date'].dt.dayofweek; b['month']=b['Date'].dt.month; b['dom']=b['Date'].dt.day
    def yoy(col):
        m=(b.assign(year=b['Date'].dt.year).groupby(['year','month'])[col].mean()
           .reset_index().pivot(index='month',columns='year',values=col).sort_index(axis=1))
        av=m.pct_change(axis=1,fill_method=None).replace([np.inf,-np.inf],np.nan).mean(axis=1).fillna(0)
        return {int(k):float(v) for k,v in av.items()}
    return {
        'rev_mean_dow_month':  b.groupby(['dow','month'])['Revenue'].mean().to_dict(),
        'cogs_mean_dow_month': b.groupby(['dow','month'])['COGS'].mean().to_dict(),
        'rev_std_month':  b.groupby('month')['Revenue'].std().fillna(0).to_dict(),
        'cogs_std_month': b.groupby('month')['COGS'].std().fillna(0).to_dict(),
        'rev_median_day':  b.groupby('dom')['Revenue'].median().to_dict(),
        'cogs_median_day': b.groupby('dom')['COGS'].median().to_dict(),
        'rev_yoy_growth_month':  yoy('Revenue'),
        'cogs_yoy_growth_month': yoy('COGS'),
    }

def _validate_continuity(df):
    expected=pd.date_range(df['Date'].min(),df['Date'].max(),freq='D')
    missing=expected.difference(pd.DatetimeIndex(df['Date']))
    if len(missing)>0:
        raise ValueError(f'Sales has {len(missing)} missing dates. First 5: {missing[:5].tolist()}')

def build_features(df, statistics=None, target_col='Revenue'):
    base=df.copy(); base['Date']=pd.to_datetime(base['Date'])
    base=base.sort_values('Date').reset_index(drop=True)
    _validate_continuity(base)
    feats=pd.DataFrame({'Date':base['Date']}); dates=base['Date']
    feats['day_of_week']=dates.dt.dayofweek
    feats['month']=dates.dt.month
    feats['day_of_month']=dates.dt.day
    feats['is_payday_window']=(feats['day_of_month']>=25).astype(int)
    feats['dist_to_payday']=dates.apply(_calc_dist_to_payday)
    feats['is_weekend']=(feats['day_of_week']>=5).astype(int)
    feats['days_to_tet']=dates.apply(_calc_days_to_tet)
    feats['is_tet_buildup']=feats['days_to_tet'].between(-21,-11).astype(int)
    feats['is_tet_peak']=feats['days_to_tet'].between(-10,-4).astype(int)
    feats['is_tet_holiday']=feats['days_to_tet'].between(-3,3).astype(int)
    feats['is_tet_reopening']=feats['days_to_tet'].between(4,10).astype(int)
    feats['is_gift_peak']=dates.apply(_is_gift_peak).astype(int)
    feats['is_travel_peak']=dates.apply(_is_travel_peak).astype(int)
    feats['is_year_end_festive']=((dates.dt.month==12)&dates.dt.day.between(10,30)).astype(int)
    feats['is_ghost_month']=dates.apply(_is_ghost_month).astype(int)
    feats['dist_to_nearest_holiday']=dates.apply(_calc_dist_to_nearest_holiday)
    feats['shopping_festival_score']=dates.apply(_calc_shopping_festival_score)
    feats['days_to_next_big_sale']=dates.apply(_calc_days_to_next_big_sale)
    feats['is_sale_leadup']=feats['days_to_next_big_sale'].between(1,3).astype(int)
    feats['is_black_friday']=dates.apply(lambda d: d.normalize()==_get_black_friday(d.year)).astype(int)
    feats['is_cyber_monday']=dates.apply(lambda d: d.normalize()==_get_cyber_monday(d.year)).astype(int)
    feats['rev_lag_364']=base['Revenue'].shift(364)
    feats['cogs_lag_364']=base['COGS'].shift(364)
    feats['rev_lag_728']=base['Revenue'].shift(728)
    feats['cogs_lag_728']=base['COGS'].shift(728)
    feats['rev_roll_mean_28_lag_728']=base['Revenue'].shift(728).rolling(28,center=True,min_periods=1).mean()
    feats['cogs_roll_mean_28_lag_728']=base['COGS'].shift(728).rolling(28,center=True,min_periods=1).mean()
    stats=statistics or _compute_statistics(base)
    rev_def=float(base['Revenue'].mean()); cogs_def=float(base['COGS'].mean())
    feats['stat_rev_mean_dow_month']=feats.apply(
        lambda r: stats['rev_mean_dow_month'].get((r['day_of_week'],r['month']),rev_def),axis=1)
    feats['stat_cogs_mean_dow_month']=feats.apply(
        lambda r: stats['cogs_mean_dow_month'].get((r['day_of_week'],r['month']),cogs_def),axis=1)
    feats['stat_rev_std_month']=feats['month'].map(stats['rev_std_month']).fillna(float(base['Revenue'].std()))
    feats['stat_cogs_std_month']=feats['month'].map(stats['cogs_std_month']).fillna(float(base['COGS'].std()))
    feats['stat_rev_median_day']=feats['day_of_month'].map(stats['rev_median_day']).fillna(float(base['Revenue'].median()))
    feats['stat_cogs_median_day']=feats['day_of_month'].map(stats['cogs_median_day']).fillna(float(base['COGS'].median()))
    feats['stat_rev_yoy_growth_month']=feats['month'].map(stats['rev_yoy_growth_month']).fillna(0.0)
    feats['stat_cogs_yoy_growth_month']=feats['month'].map(stats['cogs_yoy_growth_month']).fillna(0.0)
    doy=dates.dt.dayofyear
    feats['sin_annual_1']=np.sin(2*np.pi*doy/365.25)
    feats['cos_annual_1']=np.cos(2*np.pi*doy/365.25)
    cv_regs=_create_covid_regressors(dates)
    for col in cv_regs.columns: feats[col]=cv_regs[col].values
    feats['is_tiktok_era']=(dates>=pd.Timestamp('2022-04-28')).astype(int)
    feats['year']=dates.dt.year
    feats['target']=base[target_col].values
    return feats[['Date',*FEATURE_COLS,'target']]

print('✅ Feature engineering helpers loaded.')


✅ Feature engineering helpers loaded.


In [ ]:
# Cell 5 — Load data
sales      = pd.read_csv(DATA_DIR / 'sales.csv',             parse_dates=['Date'])
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv', parse_dates=['Date'])

print(f'Sales rows: {len(sales):,}  |  {sales["Date"].min().date()} → {sales["Date"].max().date()}')
print(f'Submission rows: {len(sample_sub):,}  |  {sample_sub["Date"].min().date()} → {sample_sub["Date"].max().date()}')

TEST_DATES = pd.DatetimeIndex(sample_sub['Date'])
assert len(TEST_DATES) == HORIZON, f'Expected {HORIZON} test dates, got {len(TEST_DATES)}'
print(f'✅ Test horizon: {HORIZON} days')


Sales rows: 3,833  |  2012-07-04 → 2022-12-31
Submission rows: 548  |  2023-01-01 → 2024-07-01
✅ Test horizon: 548 days


In [ ]:
# Cell 6 — Prepare train features (anti-leakage: statistics on full train only)
print('Computing train statistics...')
train_stats = _compute_statistics(sales)

print('Building train feature matrix...')
train_features = build_features(sales, statistics=train_stats, target_col='Revenue')
train_clean     = train_features.dropna(subset=['rev_lag_364','rev_lag_728']).reset_index(drop=True)

X_train_full     = train_clean[FEATURE_COLS]
y_train_rev      = train_clean['target']
train_dates_full = pd.DatetimeIndex(train_clean['Date'])

print(f'Train rows after lag warmup: {len(train_clean):,}')
print(f'Train period: {train_dates_full.min().date()} → {train_dates_full.max().date()}')
print(f'NaN in X_train: {X_train_full.isna().sum().sum()}')


Computing train statistics...
Building train feature matrix...
Train rows after lag warmup: 3,105
Train period: 2014-07-02 → 2022-12-31
NaN in X_train: 0


In [ ]:
# Cell 7 — Adversarial validation
print('Building test features for adversarial validation...')
test_placeholder = pd.DataFrame({
    'Date':    TEST_DATES,
    'Revenue': np.full(HORIZON, sales['Revenue'].mean()),
    'COGS':    np.full(HORIZON, sales['COGS'].mean()),
})
combined_adv    = pd.concat([sales, test_placeholder], ignore_index=True)
all_feats_adv   = build_features(combined_adv, statistics=train_stats, target_col='Revenue')
test_feats_adv  = all_feats_adv.iloc[-HORIZON:][FEATURE_COLS].reset_index(drop=True)

X_combined = pd.concat([X_train_full, test_feats_adv], ignore_index=True)
y_adv      = np.concatenate([np.zeros(len(X_train_full)), np.ones(HORIZON)])

clf_adv = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
clf_adv.fit(X_combined, y_adv)
adv_auc = roc_auc_score(y_adv, clf_adv.predict_proba(X_combined)[:,1])

TEMPORAL_COLS = {'year','month','day_of_month','day_of_week',
                 'days_to_tet','dist_to_payday','days_to_next_big_sale',
                 'dist_to_nearest_holiday','sin_annual_1','cos_annual_1'}
non_temporal = [c for c in FEATURE_COLS if c not in TEMPORAL_COLS]
clf_adv2 = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
clf_adv2.fit(X_combined[non_temporal], y_adv)
adv_auc_nt = roc_auc_score(y_adv, clf_adv2.predict_proba(X_combined[non_temporal])[:,1])

print(f'\n=== ADVERSARIAL VALIDATION ===')
print(f'AUC (all features):      {adv_auc:.3f}  ← always high for time series, ignore')
print(f'AUC (non-temporal only): {adv_auc_nt:.3f}  ← this matters')
if adv_auc_nt > 0.85:
    print('[!!] HIGH SHIFT — check lag feature distributions')
elif adv_auc_nt > 0.70:
    print('[!]  Moderate shift — acceptable for long horizon')
else:
    print('[OK] Non-temporal distributions similar — CV trustworthy')
print('\n✅ Adversarial validation done.')


Building test features for adversarial validation...

=== ADVERSARIAL VALIDATION ===
AUC (all features):      1.000  ← always high for time series, ignore
AUC (non-temporal only): 1.000  ← this matters
[!!] HIGH SHIFT — check lag feature distributions

✅ Adversarial validation done.


In [ ]:
# Cell 8 — LightGBM params + CV/store initialisation
# !! oof_store, test_store, cv_results are ONLY initialised here, never reset !!

LGB_BASE = {
    'verbose': -1, 'n_jobs': -1, 'random_state': 42,
    'n_estimators': 5000,
    'learning_rate': 0.03,
    'num_leaves': 127, 'max_depth': -1, 'min_child_samples': 30,
    'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'reg_alpha': 0.05, 'reg_lambda': 0.1,
}
LGB_HUBER   = {**LGB_BASE, 'objective': 'huber',         'alpha': 0.9,              'metric': 'mae'}
LGB_L1      = {**LGB_BASE, 'objective': 'regression_l1',                             'metric': 'mae'}
LGB_TWEEDIE = {**LGB_BASE, 'objective': 'tweedie',       'tweedie_variance_power': 1.5, 'metric': 'mae'}

tscv = TimeSeriesSplit(n_splits=5, gap=30, test_size=180)

# Global prediction stores — populated incrementally, never reset
oof_store  = {}   # model_name → OOF array (len = len(train_clean))
test_store = {}   # model_name → test array (len = HORIZON)
cv_results = []   # list of dicts for the final CV table

def assert_no_leakage(X_tr, X_val, y_tr, y_val, fold_idx):
    tr_max  = pd.DatetimeIndex(X_tr['Date'] if 'Date' in X_tr.columns else X_tr.index).max()
    val_min = pd.DatetimeIndex(X_val['Date'] if 'Date' in X_val.columns else X_val.index).min()
    assert tr_max < val_min, f'FOLD {fold_idx} LEAKAGE: train_max={tr_max} >= val_min={val_min}'
    print(f'  [OK] Fold {fold_idx} leakage check: train→{tr_max.date()}, val from {val_min.date()}')

print('✅ LightGBM params + CV setup ready.')
print(f'   Models: LightGBM Huber / L1 / Tweedie')
print(f'   CV:     TimeSeriesSplit(n_splits=5, gap=30, test_size=180)')


✅ LightGBM params + CV setup ready.
   Models: LightGBM Huber / L1 / Tweedie
   CV:     TimeSeriesSplit(n_splits=5, gap=30, test_size=180)


In [ ]:
# Cell 9 — CV helper + fast_recursive_forecast
#
# fast_recursive_forecast:
#   Calls build_features ONCE for the full test period, then patches
#   rev_lag_364 in-place for the last ~184 test days where the lag points
#   into the test period. ~50x faster than the naive per-step approach.

def fast_recursive_forecast(m, sales_df, test_dates_seq, train_stats,
                             use_log=False, res_var=0.0):
    n       = len(test_dates_seq)
    test_ts = pd.to_datetime(list(test_dates_seq))

    # 1. Build extended df with placeholder Revenue for test days
    placeholder = pd.DataFrame({
        'Date':    test_ts,
        'Revenue': float(sales_df['Revenue'].mean()),
        'COGS':    np.nan,   # unknown; LightGBM handles NaN fine
    })
    ext = pd.concat([sales_df[['Date','Revenue','COGS']], placeholder],
                    ignore_index=True)

    # 2. Compute ALL features in ONE call (lag_728 always points to train; safe)
    feats_all = build_features(ext, statistics=train_stats, target_col='Revenue')
    X = feats_all.iloc[-n:][FEATURE_COLS].copy().reset_index(drop=True).astype(float)

    # 3. Date → index lookup for recursive patch
    date_to_i = {d.normalize(): i for i, d in enumerate(test_ts)}

    preds = np.zeros(n)
    for i in range(n):
        # Patch rev_lag_364 when lag points into test period (days 364+)
        lag364 = (test_ts[i] - pd.Timedelta(days=364)).normalize()
        j = date_to_i.get(lag364)
        if j is not None:               # j < i always (lag is in the past)
            X.loc[i, 'rev_lag_364']  = preds[j]
            X.loc[i, 'cogs_lag_364'] = np.nan   # COGS unknown for test period

        raw = float(m.predict(X.iloc[[i]])[0])
        preds[i] = max(0.0, float(np.expm1(raw + res_var/2)) if use_log else raw)

    return preds


def fast_recursive_cogs_forecast(m_cogs, sales_df, test_dates_seq,
                                  train_stats, revenue_preds):
    """Same optimisation for COGS model; uses predicted Revenue as input."""
    n       = len(test_dates_seq)
    test_ts = pd.to_datetime(list(test_dates_seq))

    placeholder = pd.DataFrame({
        'Date':    test_ts,
        'Revenue': revenue_preds.astype(float),
        'COGS':    np.nan,
    })
    ext = pd.concat([sales_df[['Date','Revenue','COGS']], placeholder],
                    ignore_index=True)

    feats_all = build_features(ext, statistics=train_stats, target_col='COGS')
    X = feats_all.iloc[-n:][FEATURE_COLS].copy().reset_index(drop=True).astype(float)

    date_to_i = {d.normalize(): i for i, d in enumerate(test_ts)}
    cogs_preds = np.zeros(n)

    for i in range(n):
        lag364 = (test_ts[i] - pd.Timedelta(days=364)).normalize()
        j = date_to_i.get(lag364)
        if j is not None:
            X.loc[i, 'cogs_lag_364'] = cogs_preds[j]
            X.loc[i, 'rev_lag_364']  = revenue_preds[j]

        cogs_preds[i] = max(0.0, float(m_cogs.predict(X.iloc[[i]])[0]))

    return cogs_preds


def run_lgb_cv(model_name, params, use_log_target, seeds=(42, 123, 7)):
    """5-fold TimeSeriesSplit CV with seed bagging for one LightGBM variant."""
    print(f'\n=== {model_name} ===')
    oof = np.zeros(len(train_clean))
    fold_maes = []

    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train_full), 1):
        Xf_tr  = train_clean.iloc[tr_idx]
        Xf_val = train_clean.iloc[val_idx]
        y_tr   = y_train_rev.iloc[tr_idx]
        y_val  = y_train_rev.iloc[val_idx]
        assert_no_leakage(Xf_tr, Xf_val, y_tr, y_val, fold)

        val_preds_seeds = []
        for seed in seeds:
            sp = {**params, 'random_state': seed}
            if use_log_target:
                y_log     = np.log1p(y_tr.values)
                y_val_log = np.log1p(y_val.values)
                m = lgb.LGBMRegressor(**sp)
                m.fit(Xf_tr[FEATURE_COLS], y_log,
                      eval_set=[(Xf_val[FEATURE_COLS], y_val_log)],
                      callbacks=[lgb.early_stopping(200,verbose=False), lgb.log_evaluation(0)])
                res_var = float(np.var(y_log - m.predict(Xf_tr[FEATURE_COLS])))
                vp = np.expm1(m.predict(Xf_val[FEATURE_COLS]) + res_var/2)
            else:
                m = lgb.LGBMRegressor(**sp)
                m.fit(Xf_tr[FEATURE_COLS], y_tr.values,
                      eval_set=[(Xf_val[FEATURE_COLS], y_val.values)],
                      callbacks=[lgb.early_stopping(200,verbose=False), lgb.log_evaluation(0)])
                vp = m.predict(Xf_val[FEATURE_COLS])
            val_preds_seeds.append(np.clip(vp, 0, None))

        oof[val_idx] = np.mean(val_preds_seeds, axis=0)
        fold_mae = mean_absolute_error(y_val, oof[val_idx])
        fold_maes.append(fold_mae)
        print(f'  Fold {fold} MAE: {fold_mae:,.0f} VND')

    all_val_idx = []
    for _, val_idx in tscv.split(X_train_full):
        all_val_idx.extend(val_idx)
    all_val_idx = sorted(set(all_val_idx))

    oof_covered   = oof[all_val_idx]
    actual_covered = y_train_rev.values[all_val_idx]

    avg_mae  = float(np.mean(fold_maes))
    avg_rmse = float(np.sqrt(mean_squared_error(actual_covered, oof_covered)))
    r2 = float(1 - np.sum((actual_covered - oof_covered)**2) /
                  np.sum((actual_covered - actual_covered.mean())**2))
    print(f'  Avg MAE: {avg_mae:,.0f} | RMSE: {avg_rmse:,.0f} | R2: {r2:.3f}')

    cv_results.append({'model': model_name, 'fold_maes': fold_maes,
                       'avg_mae': avg_mae, 'avg_rmse': avg_rmse, 'r2': r2})
    oof_store[model_name] = oof
    return oof

print('✅ CV helpers + fast_recursive_forecast ready.')


✅ CV helpers + fast_recursive_forecast ready.


In [ ]:
# Cell 10 — LightGBM Huber CV (log1p target + Jensen correction)
oof_huber = run_lgb_cv('LightGBM Huber', LGB_HUBER, use_log_target=True)


=== LightGBM Huber ===
  [OK] Fold 1 leakage check: train→2020-06-14, val from 2020-07-15
  Fold 1 MAE: 567,657 VND
  [OK] Fold 2 leakage check: train→2020-12-11, val from 2021-01-11
  Fold 2 MAE: 791,508 VND
  [OK] Fold 3 leakage check: train→2021-06-09, val from 2021-07-10
  Fold 3 MAE: 554,958 VND
  [OK] Fold 4 leakage check: train→2021-12-06, val from 2022-01-06
  Fold 4 MAE: 750,914 VND
  [OK] Fold 5 leakage check: train→2022-06-04, val from 2022-07-05
  Fold 5 MAE: 541,291 VND
  Avg MAE: 641,266 | RMSE: 948,253 | R2: 0.660


In [ ]:
# Cell 11 — LightGBM L1/MAE CV
oof_l1 = run_lgb_cv('LightGBM L1', LGB_L1, use_log_target=False)


=== LightGBM L1 ===
  [OK] Fold 1 leakage check: train→2020-06-14, val from 2020-07-15
  Fold 1 MAE: 485,120 VND
  [OK] Fold 2 leakage check: train→2020-12-11, val from 2021-01-11
  Fold 2 MAE: 647,525 VND
  [OK] Fold 3 leakage check: train→2021-06-09, val from 2021-07-10
  Fold 3 MAE: 549,822 VND
  [OK] Fold 4 leakage check: train→2021-12-06, val from 2022-01-06
  Fold 4 MAE: 700,084 VND
  [OK] Fold 5 leakage check: train→2022-06-04, val from 2022-07-05
  Fold 5 MAE: 531,669 VND
  Avg MAE: 582,844 | RMSE: 833,997 | R2: 0.737


In [ ]:
# Cell 12 — LightGBM Tweedie CV
oof_tweedie = run_lgb_cv('LightGBM Tweedie', LGB_TWEEDIE, use_log_target=False)


=== LightGBM Tweedie ===
  [OK] Fold 1 leakage check: train→2020-06-14, val from 2020-07-15
  Fold 1 MAE: 543,895 VND
  [OK] Fold 2 leakage check: train→2020-12-11, val from 2021-01-11
  Fold 2 MAE: 756,837 VND
  [OK] Fold 3 leakage check: train→2021-06-09, val from 2021-07-10
  Fold 3 MAE: 538,177 VND
  [OK] Fold 4 leakage check: train→2021-12-06, val from 2022-01-06
  Fold 4 MAE: 722,702 VND
  [OK] Fold 5 leakage check: train→2022-06-04, val from 2022-07-05
  Fold 5 MAE: 544,078 VND
  Avg MAE: 621,138 | RMSE: 913,981 | R2: 0.684


In [ ]:
# Cell 13 — Final LightGBM fit on full train + optimised test forecasts
# Uses fast_recursive_forecast (~50x faster than naive per-step approach)
print('Fitting final LightGBM models on full train...')
final_models = {}   # model_name → model object (used by SHAP cell later)

for model_name, params, use_log in [
    ('LightGBM Huber',   LGB_HUBER,   True),
    ('LightGBM L1',      LGB_L1,      False),
    ('LightGBM Tweedie', LGB_TWEEDIE, False),
]:
    print(f'\n  [{model_name}]')
    seed_test_preds = []

    for seed in (42, 123, 7):
        sp = {**params, 'random_state': seed}
        if use_log:
            y_log = np.log1p(y_train_rev.values)
            m = lgb.LGBMRegressor(**sp)
            m.fit(X_train_full[FEATURE_COLS], y_log,
                  eval_set=[(X_train_full[FEATURE_COLS].iloc[-180:], y_log[-180:])],
                  callbacks=[lgb.early_stopping(200,verbose=False), lgb.log_evaluation(0)])
            res_var = float(np.var(y_log - m.predict(X_train_full[FEATURE_COLS])))
        else:
            m = lgb.LGBMRegressor(**sp)
            m.fit(X_train_full[FEATURE_COLS], y_train_rev.values,
                  eval_set=[(X_train_full[FEATURE_COLS].iloc[-180:], y_train_rev.values[-180:])],
                  callbacks=[lgb.early_stopping(200,verbose=False), lgb.log_evaluation(0)])
            res_var = 0.0

        preds = fast_recursive_forecast(m, sales, TEST_DATES, train_stats,
                                        use_log=use_log, res_var=res_var)
        seed_test_preds.append(preds)

    test_store[model_name] = np.mean(seed_test_preds, axis=0)
    final_models[model_name] = m   # last seed — good enough for SHAP
    print(f'  → test mean: {test_store[model_name].mean():,.0f} VND/day')

print('\n✅ LightGBM final fit complete.')


Fitting final LightGBM models on full train...

  [LightGBM Huber]
  → test mean: 3,091,046 VND/day

  [LightGBM L1]
  → test mean: 3,266,015 VND/day

  [LightGBM Tweedie]
  → test mean: 3,140,419 VND/day

✅ LightGBM final fit complete.


In [ ]:
# Cell 14 — Save LightGBM predictions to Drive
for name in ['LightGBM Huber', 'LightGBM L1', 'LightGBM Tweedie']:
    safe = name.replace(' ','_').lower()
    np.save(OUTPUT_DIR / f'{safe}_test.npy', test_store[name])
    np.save(OUTPUT_DIR / f'{safe}_oof.npy',  oof_store[name])
print('[OK] LightGBM predictions saved to', OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.glob('*.npy')): print(f'  {f.name}')


[OK] LightGBM predictions saved to /content/drive/MyDrive/datathon
  lightgbm_huber_oof.npy
  lightgbm_huber_test.npy
  lightgbm_l1_oof.npy
  lightgbm_l1_test.npy
  lightgbm_tweedie_oof.npy
  lightgbm_tweedie_test.npy


In [ ]:
# Cell 15 — Prophet CV + final fit
from prophet import Prophet
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

holidays_df = _create_holidays_df()

def _run_prophet_fold(train_df, val_dates):
    regs = _create_covid_regressors(train_df['ds'])
    tr   = pd.concat([train_df.reset_index(drop=True), regs], axis=1)
    m = Prophet(yearly_seasonality=10, weekly_seasonality=True, daily_seasonality=False,
                seasonality_mode='multiplicative', changepoint_prior_scale=0.05,
                seasonality_prior_scale=15.0, holidays_prior_scale=20.0,
                holidays=holidays_df, interval_width=0.95)
    for col in regs.columns: m.add_regressor(col)
    m.fit(tr)
    future = pd.concat([pd.DataFrame({'ds': val_dates}).reset_index(drop=True),
                        _create_covid_regressors(val_dates)], axis=1)
    return np.clip(m.predict(future)['yhat'].values, 0, None)

print('Running Prophet CV (5 folds)...')
prophet_df   = sales.rename(columns={'Date':'ds','Revenue':'y'})[['ds','y']]
oof_prophet  = np.zeros(len(train_clean))
fold_maes_p  = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train_full), 1):
    tr_dates  = train_clean['Date'].iloc[tr_idx]
    val_dates = pd.DatetimeIndex(train_clean['Date'].iloc[val_idx])
    fold_train = prophet_df[prophet_df['ds'].isin(tr_dates)]
    y_val = y_train_rev.iloc[val_idx].values
    vp = _run_prophet_fold(fold_train, val_dates)
    oof_prophet[val_idx] = vp
    fold_maes_p.append(mean_absolute_error(y_val, vp))
    print(f'  Fold {fold} MAE: {fold_maes_p[-1]:,.0f}')

avg_mae_p  = float(np.mean(fold_maes_p))
avg_rmse_p = float(np.sqrt(mean_squared_error(y_train_rev.values, oof_prophet)))
r2_p = float(1 - np.sum((y_train_rev.values - oof_prophet)**2) /
                 np.sum((y_train_rev.values - y_train_rev.mean())**2))
print(f'Prophet CV: MAE={avg_mae_p:,.0f} | RMSE={avg_rmse_p:,.0f} | R2={r2_p:.3f}')
cv_results.append({'model':'Prophet','fold_maes':fold_maes_p,
                   'avg_mae':avg_mae_p,'avg_rmse':avg_rmse_p,'r2':r2_p})

print('\nFitting final Prophet on full train...')
full_regs        = _create_covid_regressors(prophet_df['ds'])
full_train_p     = pd.concat([prophet_df.reset_index(drop=True), full_regs], axis=1)
m_prophet_final  = Prophet(yearly_seasonality=10, weekly_seasonality=True, daily_seasonality=False,
                            seasonality_mode='multiplicative', changepoint_prior_scale=0.05,
                            seasonality_prior_scale=15.0, holidays_prior_scale=20.0,
                            holidays=holidays_df, interval_width=0.95)
for col in full_regs.columns: m_prophet_final.add_regressor(col)
m_prophet_final.fit(full_train_p)

test_future = pd.concat([pd.DataFrame({'ds': TEST_DATES}).reset_index(drop=True),
                          _create_covid_regressors(TEST_DATES)], axis=1)
test_preds_prophet = np.clip(m_prophet_final.predict(test_future)['yhat'].values, 0, None)

test_store['Prophet'] = test_preds_prophet
oof_store['Prophet']  = oof_prophet
print(f'Prophet test mean: {test_preds_prophet.mean():,.0f} VND/day')


Running Prophet CV (5 folds)...
  Fold 1 MAE: 651,062
  Fold 2 MAE: 901,059
  Fold 3 MAE: 677,350
  Fold 4 MAE: 1,037,837
  Fold 5 MAE: 584,306
Prophet CV: MAE=770,323 | RMSE=4,647,828 | R2=-2.058

Fitting final Prophet on full train...
Prophet test mean: 3,486,923 VND/day


In [ ]:
# Cell 16 — Save Prophet predictions to Drive
np.save(OUTPUT_DIR / 'prophet_test.npy', test_preds_prophet)
np.save(OUTPUT_DIR / 'prophet_oof.npy',  oof_prophet)
print('[OK] Prophet predictions saved to', OUTPUT_DIR)


[OK] Prophet predictions saved to /content/drive/MyDrive/datathon


In [ ]:
# Cell 17 — GPU verification
import subprocess, torch

result = subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'],
    capture_output=True, text=True)
print('nvidia-smi:', result.stdout.strip() or '[not available]')

assert torch.cuda.is_available(), (
    'GPU not detected! Go to: Runtime > Change runtime type > T4 GPU > Save')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[OK] GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')

USE_NBEATS_FALLBACK = vram_gb < 10
if USE_NBEATS_FALLBACK:
    print('[!] VRAM < 10 GB — using fallback N-BEATS config (smaller MLP)')
else:
    print('[OK] VRAM sufficient for full N-BEATS config.')


nvidia-smi: Tesla T4, 15360 MiB, 14913 MiB
[OK] GPU: Tesla T4  |  VRAM: 14.6 GB
[OK] VRAM sufficient for full N-BEATS config.


In [ ]:
# Cell 18 — N-BEATS training  (oof_store / test_store / cv_results NOT reset here)
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE as NF_MAE

if USE_NBEATS_FALLBACK:
    mlp_units, batch_size, max_steps = [[256,256],[256,256]], 16, 300
    print('[!] N-BEATS fallback config: mlp_units=256, batch_size=16, max_steps=300')
else:
    mlp_units, batch_size, max_steps = [[512,512],[512,512]], 32, 500
    print('[OK] N-BEATS full config: mlp_units=512, batch_size=32, max_steps=500')

nbeats_train_df = pd.DataFrame({
    'ds':        pd.to_datetime(sales['Date']),
    'y':         sales['Revenue'].astype(float),
    'unique_id': 'total_revenue',
})

nf_model = NeuralForecast(
    models=[NBEATS(
        h=HORIZON,
        input_size=2*365,
        stack_types=['trend','seasonality'],
        n_blocks=[3,3],
        mlp_units=mlp_units,
        n_harmonics=2,
        n_polynomials=2,
        dropout_prob_theta=0,   # dropout not supported — must be 0
        learning_rate=1e-3,
        max_steps=max_steps,
        batch_size=batch_size,
        random_seed=42,
        loss=NF_MAE(),
    )],
    freq='D',
)

print(f'Training N-BEATS (max_steps={max_steps})...')
nf_model.fit(nbeats_train_df)

forecast_df       = nf_model.predict(nbeats_train_df)
test_preds_nbeats = np.clip(forecast_df['NBEATS'].values, 0, None)
test_store['N-BEATS'] = test_preds_nbeats

print('Running N-BEATS cross_validation for OOF...')
try:
    cv_df = nf_model.cross_validation(nbeats_train_df, n_windows=3, step_size=180)
    cv_merged = train_clean[['Date']].merge(
        cv_df.rename(columns={'ds':'Date','NBEATS':'pred'})[['Date','pred']]
              .groupby('Date').mean().reset_index(),
        on='Date', how='left')
    oof_nbeats = cv_merged['pred'].fillna(y_train_rev.mean()).values
    valid_mask = oof_nbeats != y_train_rev.mean()
    if valid_mask.sum() > 0:
        nbeats_mae = mean_absolute_error(y_train_rev.values[valid_mask], oof_nbeats[valid_mask])
        print(f'N-BEATS CV MAE (approx): {nbeats_mae:,.0f}')
except Exception as e:
    print(f'[!] N-BEATS OOF failed ({e}), using mean fill')
    oof_nbeats = np.full(len(train_clean), y_train_rev.mean())

oof_store['N-BEATS'] = oof_nbeats
print(f'N-BEATS test mean: {test_preds_nbeats.mean():,.0f} VND/day')
print(f'\nAll stores: oof_store={list(oof_store)}, test_store={list(test_store)}')


INFO:lightning_fabric.utilities.seed:Seed set to 42


[OK] N-BEATS full config: mlp_units=512, batch_size=32, max_steps=500


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Training N-BEATS (max_steps=500)...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

N-BEATS CV MAE (approx): 974,464
N-BEATS test mean: 2,952,580 VND/day

All stores: oof_store=['LightGBM Huber', 'LightGBM Tweedie', 'LightGBM L1', 'Prophet', 'N-BEATS'], test_store=['LightGBM Huber', 'LightGBM L1', 'LightGBM Tweedie', 'Prophet', 'N-BEATS']


In [ ]:
# Cell 19 — Save N-BEATS predictions to Drive
np.save(OUTPUT_DIR / 'nbeats_test.npy', test_preds_nbeats)
np.save(OUTPUT_DIR / 'nbeats_oof.npy',  oof_nbeats)
print('[OK] N-BEATS predictions saved to', OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.glob('*.npy')): print(f'  {f.name}')


[OK] N-BEATS predictions saved to /content/drive/MyDrive/datathon
  lightgbm_huber_oof.npy
  lightgbm_huber_test.npy
  lightgbm_l1_oof.npy
  lightgbm_l1_test.npy
  lightgbm_tweedie_oof.npy
  lightgbm_tweedie_test.npy
  nbeats_oof.npy
  nbeats_test.npy
  prophet_oof.npy
  prophet_test.npy


In [ ]:
# Chạy cell debug này trước
print(type(tscv))
print(tscv)
print(f'n_splits: {tscv.n_splits}')
print()
for i, (train_idx, val_idx) in enumerate(tscv.split(X_train_full)):
    print(f'Fold {i+1}: train={len(train_idx)} | val={len(val_idx)} '
          f'| val range: [{val_idx[0]}:{val_idx[-1]}]')

<class 'sklearn.model_selection._split.TimeSeriesSplit'>
TimeSeriesSplit(gap=30, max_train_size=None, n_splits=5, test_size=180)
n_splits: 5

Fold 1: train=2175 | val=180 | val range: [2205:2384]
Fold 2: train=2355 | val=180 | val range: [2385:2564]
Fold 3: train=2535 | val=180 | val range: [2565:2744]
Fold 4: train=2715 | val=180 | val range: [2745:2924]
Fold 5: train=2895 | val=180 | val range: [2925:3104]


In [ ]:
assert 'tscv' in dir(), "tscv chưa được khởi tạo — chạy lại Cell 8 trước"
# Cell 20 — SLSQP ensemble
MODEL_NAMES = ['LightGBM Huber', 'LightGBM L1', 'LightGBM Tweedie', 'Prophet', 'N-BEATS']

# Verify all models are present
missing = [n for n in MODEL_NAMES if n not in oof_store or n not in test_store]
assert not missing, f'Missing predictions for: {missing}'

# Fix Cell 20: chỉ dùng covered rows để tính SLSQP weights
all_val_idx = sorted(set(
    idx for _, val_idx in tscv.split(X_train_full) for idx in val_idx
))

print(f'Covered rows: {len(all_val_idx)}/{len(y_train_rev)} '
      f'({100*len(all_val_idx)/len(y_train_rev):.1f}%)')
# Kỳ vọng: ~60-80% tùy TimeSeriesSplit config

oof_matrix_covered  = np.column_stack([oof_store[n][all_val_idx]  for n in MODEL_NAMES])
test_matrix         = np.column_stack([test_store[n]               for n in MODEL_NAMES])
y_oof_covered       = y_train_rev.values[all_val_idx]
n_models            = len(MODEL_NAMES)

def fit_slsqp_weights(oof_preds, y_true):
    def objective(w): return np.mean(np.abs(oof_preds @ w - y_true))
    result = minimize(objective, x0=np.ones(n_models)/n_models,
                      method='SLSQP', bounds=[(0.0,1.0)]*n_models,
                      constraints=[{'type':'eq','fun': lambda w: w.sum()-1.0}],
                      options={'maxiter':1000,'ftol':1e-9})
    if not result.success:
        print(f'[!] SLSQP failed — equal weights')
        return np.ones(n_models) / n_models
    return result.x

weights      = fit_slsqp_weights(oof_matrix_covered, y_oof_covered)
blended_test = test_matrix @ weights

blended_mae  = mean_absolute_error(y_oof_covered, oof_matrix_covered @ weights)
blended_rmse = float(np.sqrt(mean_squared_error(y_oof_covered, oof_matrix_covered @ weights)))
blended_r2   = float(1 - np.sum((y_oof_covered - oof_matrix_covered @ weights)**2) /
                          np.sum((y_oof_covered - y_oof_covered.mean())**2))

print('\n=== ENSEMBLE WEIGHTS (fixed) ===')
for name, w in zip(MODEL_NAMES, weights):
    print(f'  {name:<22}: {w:.3f}')
print(f'\nBlended OOF (covered only): MAE={blended_mae:,.0f} | RMSE={blended_rmse:,.0f} | R2={blended_r2:.3f}')

test_store['Ensemble'] = blended_test
revenue_test = blended_test.copy()


Covered rows: 900/3105 (29.0%)

=== ENSEMBLE WEIGHTS (fixed) ===
  LightGBM Huber        : 0.200
  LightGBM L1           : 0.200
  LightGBM Tweedie      : 0.200
  Prophet               : 0.200
  N-BEATS               : 0.200

Blended OOF (covered only): MAE=636,416 | RMSE=929,829 | R2=0.673


In [ ]:
# Cell 20 — simplified ensemble (3 LGB only)
MODEL_NAMES = ['LightGBM Huber', 'LightGBM L1', 'LightGBM Tweedie']

cv_mae = {
    'LightGBM Huber'   : 641_266,
    'LightGBM L1'      : 582_844,
    'LightGBM Tweedie' : 621_138,
}

# Inverse-MAE weighting (no SLSQP needed for 3 similar models)
inv_mae = np.array([1/cv_mae[n] for n in MODEL_NAMES])
weights = inv_mae / inv_mae.sum()

print('=== ENSEMBLE WEIGHTS (3 LGB only) ===')
for name, w in zip(MODEL_NAMES, weights):
    print(f'  {name:<22}: {w:.3f}')

test_matrix = np.column_stack([test_store[n] for n in MODEL_NAMES])
blended_test = test_matrix @ weights

# Validate on covered OOF
all_val_idx = sorted(set(idx for _, val_idx in tscv.split(X_train_full) for idx in val_idx))
oof_matrix = np.column_stack([oof_store[n][all_val_idx] for n in MODEL_NAMES])
y_oof = y_train_rev.values[all_val_idx]
blended_oof = oof_matrix @ weights

mae = mean_absolute_error(y_oof, blended_oof)
r2 = 1 - np.sum((y_oof - blended_oof)**2) / np.sum((y_oof - y_oof.mean())**2)
print(f'\nBlended OOF: MAE={mae:,.0f} | R2={r2:.3f}')

test_store['Ensemble'] = blended_test
revenue_test = blended_test.copy()

=== ENSEMBLE WEIGHTS (3 LGB only) ===
  LightGBM Huber        : 0.319
  LightGBM L1           : 0.351
  LightGBM Tweedie      : 0.330

Blended OOF: MAE=607,807 | R2=0.703


In [ ]:
# Cell 21 — COGS strategy
last_90       = sales.sort_values('Date').tail(90)
margin_series = last_90['COGS'] / last_90['Revenue']
cv_margin     = float(margin_series.std() / margin_series.mean())
avg_margin    = float(margin_series.mean())
revenue_test  = blended_test.copy()

print(f'CV(margin) last 90 days: {cv_margin:.4f}')
print(f'Avg COGS/Revenue ratio:  {avg_margin:.4f}')

if cv_margin < 0.08:
    print('Strategy: RATIO (stable margin)')
    cogs_test = revenue_test * avg_margin

elif cv_margin < 0.20:
    print('Strategy: SEPARATE MODEL (LightGBM L1 on COGS target)')
    cogs_features = build_features(sales, statistics=train_stats, target_col='COGS')
    cogs_clean    = cogs_features.dropna(subset=['rev_lag_364','rev_lag_728']).reset_index(drop=True)
    y_cogs        = cogs_clean['target']
    m_cogs = lgb.LGBMRegressor(**{**LGB_L1, 'random_state': 42})
    m_cogs.fit(cogs_clean[FEATURE_COLS], y_cogs.values,
               eval_set=[(cogs_clean[FEATURE_COLS].iloc[-180:], y_cogs.values[-180:])],
               callbacks=[lgb.early_stopping(200,verbose=False), lgb.log_evaluation(0)])
    # Optimised recursive COGS forecast (same fast approach as Revenue)
    cogs_test = fast_recursive_cogs_forecast(
        m_cogs, sales, TEST_DATES, train_stats, revenue_test)

else:
    print('Strategy: BLEND 50/50 (volatile margin — ratio fallback)')
    cogs_test = revenue_test * avg_margin

# Hard constraint: COGS < 95% Revenue
cogs_test = np.minimum(cogs_test, 0.95 * revenue_test)
cogs_test = np.clip(cogs_test, 0, None)

print(f'COGS test mean:  {cogs_test.mean():,.0f} VND/day')
print(f'Implied margin:  {(1 - cogs_test.mean()/revenue_test.mean()):.2%}')


CV(margin) last 90 days: 0.1099
Avg COGS/Revenue ratio:  0.9165
Strategy: SEPARATE MODEL (LightGBM L1 on COGS target)
COGS test mean:  2,838,490 VND/day
Implied margin:  10.42%


In [ ]:
# Cell 22 — SHAP analysis
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

best_model = final_models['LightGBM Huber']
X_shap     = X_train_full[FEATURE_COLS]
sample_idx = np.random.RandomState(42).choice(len(X_shap), size=min(2000, len(X_shap)), replace=False)
X_sample   = X_shap.iloc[sample_idx]

print('Computing SHAP values...')
explainer  = shap.TreeExplainer(best_model, feature_perturbation='interventional',
                                 data=X_shap.sample(200, random_state=42))
shap_vals  = explainer.shap_values(X_sample)
np.save(OUTPUT_DIR / 'shap_values.npy', shap_vals)

# Plot 1: Beeswarm
plt.figure(figsize=(12,8))
shap.summary_plot(shap_vals, X_sample, max_display=20, show=False)
plt.title('SHAP Feature Importance — Revenue Forecasting (LightGBM Huber)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print('[OK] shap_beeswarm.png saved')

# Plot 2: Bar
plt.figure(figsize=(10,8))
shap.summary_plot(shap_vals, X_sample, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Mean |Value| — Top Features')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print('[OK] shap_bar.png saved')

# Plot 3: Waterfall — Tet Eve 2022
tet_eve = pd.Timestamp('2022-01-31')
train_di = pd.DatetimeIndex(train_clean['Date'])
if tet_eve in train_di:
    sample_dates = train_di[sample_idx]
    if tet_eve in sample_dates:
        pos = list(sample_dates).index(tet_eve)
        plt.figure(figsize=(10,6))
        shap.waterfall_plot(shap.Explanation(
            values=shap_vals[pos], base_values=explainer.expected_value,
            data=X_sample.iloc[pos].values, feature_names=FEATURE_COLS), show=False)
        plt.title('SHAP Waterfall — Tet Eve 2022 (2022-01-31)')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'shap_waterfall_tet_eve.png', dpi=150, bbox_inches='tight')
        plt.close()
        print('[OK] shap_waterfall_tet_eve.png saved')

# Importance CSV
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': np.abs(shap_vals).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
importance_df.to_csv(OUTPUT_DIR / 'shap_importance.csv', index=False)
print('[OK] shap_importance.csv saved')

# Business translation table
translations = {
    'rev_lag_364':            'Annual seasonality dominates — high brand predictability',
    'stat_rev_mean_dow_month':'10-year pattern lookup beats COVID-contaminated recent lags',
    'days_to_tet':            'Tet window (7-14 days before) = largest spike of year',
    'is_lockdown':            'COVID lockdown reduces revenue ~450M VND/day',
    'is_payday_window':       'End-of-month salary spike — focus campaigns on 25-31',
    'shopping_festival_score':'11/11 > 9/9 > 1/1 hierarchy matches VN market',
    'year':                   '10-year trend signal — explicit growth encoding',
    'is_tiktok_era':          'TikTok Shop structural shift from Apr 2022',
}
print('\n=== SHAP BUSINESS TRANSLATION ===')
print(f'{"Feature":<30} {"Mean |SHAP| (VND)":<22} Business Insight')
print('-'*90)
for _, row in importance_df.head(10).iterrows():
    print(f'{row["feature"]:<30} {row["mean_abs_shap"]:>18,.0f}   {translations.get(row["feature"], "--")}')


Computing SHAP values...


100%|===================| 1998/2000 [20:08<00:01]       

[OK] shap_beeswarm.png saved
[OK] shap_bar.png saved
[OK] shap_waterfall_tet_eve.png saved
[OK] shap_importance.csv saved

=== SHAP BUSINESS TRANSLATION ===
Feature                        Mean |SHAP| (VND)      Business Insight
------------------------------------------------------------------------------------------
year                                            0   10-year trend signal — explicit growth encoding
cogs_lag_364                                    0   --
stat_rev_mean_dow_month                         0   10-year pattern lookup beats COVID-contaminated recent lags
rev_roll_mean_28_lag_728                        0   --
stat_cogs_mean_dow_month                        0   --
stat_rev_median_day                             0   --
day_of_month                                    0   --
cos_annual_1                                    0   --
stat_cogs_median_day                            0   --
dist_to_payday                                  0   --


In [ ]:
# Cell 23 — Build submission + validate + save
def validate_submission(sub_df, sample_df):
    assert len(sub_df)==len(sample_df),       f'Row mismatch: {len(sub_df)} vs {len(sample_df)}'
    assert list(sub_df.columns)==['Date','Revenue','COGS'], f'Wrong columns: {sub_df.columns.tolist()}'
    assert (pd.to_datetime(sub_df['Date']).values==pd.to_datetime(sample_df['Date']).values).all(), 'Date order mismatch'
    assert not sub_df.isna().any().any(),     f'NaN found: {sub_df.isna().sum().to_dict()}'
    assert (sub_df['Revenue']>=0).all(),      f'Negative Revenue: min={sub_df["Revenue"].min():,.0f}'
    assert (sub_df['COGS']>=0).all(),         f'Negative COGS: min={sub_df["COGS"].min():,.0f}'
    viol = (sub_df['COGS'] > sub_df['Revenue']).sum()
    assert viol==0,                           f'COGS > Revenue on {viol} rows'
    rev_mean = sub_df['Revenue'].mean()
    assert 500_000 < rev_mean < 50_000_000,  f'Revenue mean suspicious: {rev_mean:,.0f}'
    print('[OK] All 7 submission checks passed')
    print(f'  Rows      : {len(sub_df)}')
    print(f'  Date range: {sub_df["Date"].min()} → {sub_df["Date"].max()}')
    print(f'  Revenue   : mean={sub_df["Revenue"].mean():,.0f}  min={sub_df["Revenue"].min():,.0f}  max={sub_df["Revenue"].max():,.0f}')
    print(f'  COGS      : mean={sub_df["COGS"].mean():,.0f}  min={sub_df["COGS"].min():,.0f}  max={sub_df["COGS"].max():,.0f}')
    print(f'  Avg margin: {(1 - sub_df["COGS"].mean()/sub_df["Revenue"].mean()):.1%}')

submission = pd.DataFrame({
    'Date':    sample_sub['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': revenue_test,
    'COGS':    cogs_test,
})
validate_submission(submission, sample_sub)
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'\n✅ submission.csv saved to {OUTPUT_DIR}')


[OK] All 7 submission checks passed
  Rows      : 548
  Date range: 2023-01-01 → 2024-07-01
  Revenue   : mean=3,168,770  min=636,682  max=10,478,775
  COGS      : mean=2,838,490  min=604,848  max=9,433,203
  Avg margin: 10.4%

✅ submission.csv saved to /content/drive/MyDrive/datathon


In [ ]:
# Cell 24 — CV results table + final summary
print('\n=== CV RESULTS PER MODEL ===')
header = f'{"Model":<22} | {"F1 MAE":>10} | {"F2 MAE":>10} | {"F3 MAE":>10} | {"F4 MAE":>10} | {"F5 MAE":>10} | {"Avg MAE":>10} | {"RMSE":>10} | {"R2":>6}'
print(header); print('-'*len(header))
rows_csv = []
for r in cv_results:
    folds = r['fold_maes']
    fs = [f'{f:>10,.0f}' if i<len(folds) else f'{"N/A":>10}' for i,f in enumerate((folds+[0]*5)[:5])]
    print(f'{r["model"]:<22} | {fs[0]} | {fs[1]} | {fs[2]} | {fs[3]} | {fs[4]} | {r["avg_mae"]:>10,.0f} | {r["avg_rmse"]:>10,.0f} | {r["r2"]:>6.3f}')
    row = {'model':r['model'],'avg_mae':r['avg_mae'],'avg_rmse':r['avg_rmse'],'r2':r['r2']}
    for i,f in enumerate(folds,1): row[f'fold_{i}_mae']=f
    rows_csv.append(row)

pd.DataFrame(rows_csv).to_csv(OUTPUT_DIR / 'cv_results.csv', index=False)

print(f'\n=== ENSEMBLE WEIGHTS ===')
for name, w in zip(MODEL_NAMES, weights):
    print(f'  {name:<22}: {w:.3f}')

print(f'\n=== SUBMISSION SUMMARY ===')
print(f'Revenue: mean={revenue_test.mean():,.0f}  min={revenue_test.min():,.0f}  max={revenue_test.max():,.0f}')
print(f'COGS:    mean={cogs_test.mean():,.0f}  min={cogs_test.min():,.0f}  max={cogs_test.max():,.0f}')
print(f'Margin:  {(1 - cogs_test.mean()/revenue_test.mean()):.1%}')
print(f'\n✅ Pipeline complete! Files in {OUTPUT_DIR}:')
for f in sorted(OUTPUT_DIR.iterdir()): print(f'  {f.name}')



=== CV RESULTS PER MODEL ===
Model                  |     F1 MAE |     F2 MAE |     F3 MAE |     F4 MAE |     F5 MAE |    Avg MAE |       RMSE |     R2
--------------------------------------------------------------------------------------------------------------------------
LightGBM Huber         |    567,657 |    791,508 |    554,958 |    750,914 |    541,291 |    641,266 |  4,636,268 | -2.043
LightGBM Tweedie       |    543,895 |    756,837 |    538,177 |    722,702 |    544,078 |    621,138 |  4,634,273 | -2.040
LightGBM L1            |    485,120 |    647,525 |    549,822 |    700,084 |    531,669 |    582,844 |  4,629,898 | -2.035
LightGBM Huber         |    567,657 |    791,508 |    554,958 |    750,914 |    541,291 |    641,266 |    948,253 |  0.660
LightGBM L1            |    485,120 |    647,525 |    549,822 |    700,084 |    531,669 |    582,844 |    833,997 |  0.737
LightGBM Tweedie       |    543,895 |    756,837 |    538,177 |    722,702 |    544,078 |    621,138 |    913